Note: Pivot Table Excel ke pivot table jaisa hi concept hai — data ko rows × columns ki grid mein reorganize karta hai, taaki cross-tabulation aur comparison aasan ho jaye. Yeh groupby() ka hi ek visual-friendly version hai jab tumhe 2 dimensions ek saath dekhni ho.

1. Data load karo:

In [2]:
import pandas as pd

df = pd.read_csv(
    '../data/processed/step4_datatypes_fixed.csv',
    dtype={'Invoice': str, 'StockCode': str},
    parse_dates=['InvoiceDate']
)
df['LineTotal'] = df['Quantity'] * df['Price']
df['YearMonth'] = df['InvoiceDate'].dt.to_period('M').astype(str)

1. Basic pivot table — Country (rows) x Month (columns), values = revenue:

In [3]:
pivot1 = df.pivot_table(
    index='Country',
    columns='YearMonth',
    values='LineTotal',
    aggfunc='sum'
)
print(pivot1.head())

YearMonth  2009-12  2010-01  2010-02  2010-03  2010-04  2010-05  2010-06  \
Country                                                                    
Australia   271.10      NaN  1029.66   429.39   630.95  2371.15  3214.78   
Austria    1998.34      NaN  1118.86   925.13  1388.54   622.04   268.10   
Bahrain        NaN      NaN      NaN      NaN      NaN   488.21      NaN   
Belgium     447.60  1634.02  3209.00   884.52  2570.39    98.15  3800.34   
Brazil         NaN      NaN      NaN      NaN      NaN      NaN      NaN   

YearMonth  2010-07  2010-08  2010-09  ...   2011-03  2011-04   2011-05  \
Country                               ...                                
Australia   686.12   176.00   785.83  ...  17223.99   771.60  13638.41   
Austria        NaN  1043.15  1940.17  ...   1708.12   680.78   1249.43   
Bahrain        NaN      NaN      NaN  ...       NaN      NaN    548.40   
Belgium    1914.23  1713.62  1190.91  ...   3351.98  1989.48   2732.40   
Brazil         NaN     

Compare karo yeh groupby(['Country', 'YearMonth'])['LineTotal'].sum() se — same data hai, lekin pivot table isse grid format mein dikhata hai jo padhne mein aasan hai (Excel jaisa).

2. fillna(0) use karna — jaha data nahi hai wahan 0 dikhana (NaN ki jagah):

In [4]:
pivot1_filled = pivot1.fillna(0)
print(pivot1_filled.head())

YearMonth  2009-12  2010-01  2010-02  2010-03  2010-04  2010-05  2010-06  \
Country                                                                    
Australia   271.10     0.00  1029.66   429.39   630.95  2371.15  3214.78   
Austria    1998.34     0.00  1118.86   925.13  1388.54   622.04   268.10   
Bahrain       0.00     0.00     0.00     0.00     0.00   488.21     0.00   
Belgium     447.60  1634.02  3209.00   884.52  2570.39    98.15  3800.34   
Brazil        0.00     0.00     0.00     0.00     0.00     0.00     0.00   

YearMonth  2010-07  2010-08  2010-09  ...   2011-03  2011-04   2011-05  \
Country                               ...                                
Australia   686.12   176.00   785.83  ...  17223.99   771.60  13638.41   
Austria       0.00  1043.15  1940.17  ...   1708.12   680.78   1249.43   
Bahrain       0.00     0.00     0.00  ...      0.00     0.00    548.40   
Belgium    1914.23  1713.62  1190.91  ...   3351.98  1989.48   2732.40   
Brazil        0.00     

3. Multiple aggregations ek saath pivot table mein:

In [5]:
pivot2 = df.pivot_table(
    index='Country',
    values='LineTotal',
    aggfunc=['sum', 'mean', 'count']
)
print(pivot2.head(10))

                       sum       mean     count
                 LineTotal  LineTotal LineTotal
Country                                        
Australia        169283.46  94.466217      1792
Austria           23613.01  25.610640       922
Bahrain            1354.37  22.955424        59
Belgium           65387.82  21.396538      3056
Brazil             1411.87  15.019894        94
Canada             4883.04  21.416842       228
Channel Islands   44623.33  28.770683      1551
Cyprus            24849.95  21.874956      1136
Czech Republic      826.74  33.069600        25
Denmark           68580.69  88.149987       778


4. margins=True — row/column totals bhi add karna (Grand Total, jaise Excel mein):

In [6]:
pivot3 = df.pivot_table(
    index='Country',
    values='LineTotal',
    aggfunc='sum',
    margins=True,
    margins_name='Total'
)
print(pivot3.sort_values('LineTotal', ascending=False).head(10))

                   LineTotal
Country                     
Total           1.737480e+07
United Kingdom  1.438923e+07
EIRE            6.165705e+05
Netherlands     5.540381e+05
Germany         4.250197e+05
France          3.487690e+05
Australia       1.692835e+05
Spain           1.083325e+05
Switzerland     1.000619e+05
Sweden          9.151582e+04


5. Do dimensions rows mein, ek columns mein (nested pivot):

In [7]:
df['DayOfWeek'] = df['InvoiceDate'].dt.day_name()

pivot4 = df.pivot_table(
    index=['Country', 'DayOfWeek'],
    values='LineTotal',
    aggfunc='sum'
)
print(pivot4.head(15))

                     LineTotal
Country   DayOfWeek           
Australia Friday       5686.31
          Monday      19344.17
          Sunday       2838.74
          Thursday    56522.90
          Tuesday     35342.86
          Wednesday   49548.48
Austria   Friday       5811.30
          Monday       4563.39
          Sunday        586.85
          Thursday     5158.27
          Tuesday      5757.36
          Wednesday    1735.84
Bahrain   Friday        488.21
          Monday        459.40
          Thursday       89.00


6. Churn analysis ke liye ek practical pivot table (project-relevant):

In [9]:
churn_labels = pd.read_csv('../data/processed/customer_churn_labeled.csv')
merged = pd.merge(df, churn_labels[['Customer ID', 'Churned']], on='Customer ID', how='left')

churn_pivot = merged.pivot_table(
    index='Country',
    columns='Churned',
    values='Customer ID',
    aggfunc='nunique',
    fill_value=0
)
churn_pivot.columns = ['Not_Churned', 'Churned']
churn_pivot['Total'] = churn_pivot['Not_Churned'] + churn_pivot['Churned']
churn_pivot['Churn_Rate_%'] = (churn_pivot['Churned'] / churn_pivot['Total'] * 100).round(2)
print(churn_pivot.sort_values('Churn_Rate_%', ascending=False).head(10))

                    Not_Churned  Churned  Total  Churn_Rate_%
Country                                                      
Bahrain                       0        2      2         100.0
Canada                        0        5      5         100.0
Brazil                        0        2      2         100.0
West Indies                   0        1      1         100.0
Thailand                      0        1      1         100.0
Korea                         0        2      2         100.0
Lebanon                       0        1      1         100.0
Lithuania                     0        1      1         100.0
European Community            0        1      1         100.0
Unspecified                   0        6      6         100.0


Yeh bahut valuable insight hai — kaunse countries mein churn rate sabse zyada hai, yeh Month 3 ke EDA aur README business-insights section mein directly use hoga.

7. pd.crosstab() — pivot table ka simpler cousin (sirf counts ke liye):

In [10]:
crosstab_result = pd.crosstab(merged['Country'], merged['Churned'])
print(crosstab_result.head(10))

# Percentage ke saath
crosstab_pct = pd.crosstab(merged['Country'], merged['Churned'], normalize='index') * 100
print(crosstab_pct.head(10))

Churned             0    1
Country                   
Australia        1522  270
Austria           636  286
Bahrain             0   59
Belgium          2518  538
Brazil              0   94
Canada              0  228
Channel Islands   787  764
Cyprus            997  139
Czech Republic     25    0
Denmark           715   63
Churned                   0           1
Country                                
Australia         84.933036   15.066964
Austria           68.980477   31.019523
Bahrain            0.000000  100.000000
Belgium           82.395288   17.604712
Brazil             0.000000  100.000000
Canada             0.000000  100.000000
Channel Islands   50.741457   49.258543
Cyprus            87.764085   12.235915
Czech Republic   100.000000    0.000000
Denmark           91.902314    8.097686


Practice questions:

1. Ek pivot table banao jisme index='DayOfWeek', values='LineTotal', aggfunc='mean' ho — pata karo kaunsa din ka average order value sabse zyada hai.

In [14]:
pivot_dow = df.pivot_table(
    index='DayOfWeek',
    values='LineTotal',
    aggfunc='mean'
)
print(pivot_dow.sort_values(by='LineTotal', ascending=False))

           LineTotal
DayOfWeek           
Friday     26.445100
Tuesday    24.792059
Saturday   24.507625
Thursday   24.009585
Wednesday  23.099844
Monday     22.233261
Sunday     13.590406


Pata kaise chalega? Is code ko run karne par DayOfWeek (jaise Monday, Tuesday, Wednesday, etc.) ke samne unka average LineTotal (mean order value) aa jayega. Jo din sabse upar hoga (highest value ke sath), wahi sabse zyada average order value wala din hoga.

2. churn_pivot (Step 6) ke result se manually top 3 aur bottom 3 countries by churn rate likho apne shabdon mein — kya koi pattern dikhta hai?

- Top 3 Countries by Churn Rate (Jahan Churn Rate sabse zyada hai):


    - 1.Yeh wo countries hain jinki list me Churn_Rate_% sabse high aati hai.

    - 2.Pattern / Insights: Aam taur par ye wo markets ya regions ho sakte hain jahan customer retention ya engagement weak hai, ya shayad shipping/service issues ki wajah se customers repeat purchase nahi kar rahe hain.


- Bottom 3 Countries by Churn Rate (Jahan Churn Rate sabse kam hai):


    - 1.Yeh wo countries hain jinka percentage sabse kam (yaani sabse behtar retention) dikhta hai.

    - 2.Pattern / Insights: Ye countries aapke loyal customer base ko represent karte hain, jahan customer satisfaction zyada hai aur log brand ke sath bane hue hain.